# BUSI Dataset Classification with Imbalance Handling

This notebook trains and compares several methods:

1. Baseline
2. Class Weighting
3. Oversampling
4. Data Augmentation
5. Focal Loss

Evaluation uses **classification reports** and a final **F1-score comparison table**.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

## Device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Dataset Path

In [3]:
DATASET_PATH = "dataset"

## Transforms

In [4]:
basic_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

aug_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

## Load Dataset

In [5]:
dataset = datasets.ImageFolder(DATASET_PATH, transform=basic_transform)

X = np.arange(len(dataset))
y = np.array(dataset.targets)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

train_dataset = Subset(dataset, X_train)
val_dataset = Subset(dataset, X_val)
test_dataset = Subset(dataset, X_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

## Model

In [6]:
def get_model():
    model = models.resnet18(weights='DEFAULT')
    model.fc = nn.Linear(model.fc.in_features, 3)
    return model

## Training Function

In [7]:
def train_model(model, train_loader, criterion, epochs=5):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    return model

## Evaluation Function

In [8]:
def evaluate(model, loader):

    model.eval()

    preds = []
    labels_list = []

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            pred = torch.argmax(outputs, dim=1)

            preds.extend(pred.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    report = classification_report(labels_list, preds, output_dict=True)
    print(classification_report(labels_list, preds))

    return report

## 1. Baseline Model

In [9]:
model_baseline = get_model().to(device)
criterion = nn.CrossEntropyLoss()

model_baseline = train_model(model_baseline, train_loader, criterion)

test_report_base = evaluate(model_baseline, test_loader)

              precision    recall  f1-score   support

           0       0.93      0.85      0.89       134
           1       0.69      0.95      0.80        63
           2       0.89      0.62      0.74        40

    accuracy                           0.84       237
   macro avg       0.84      0.81      0.81       237
weighted avg       0.86      0.84      0.84       237



## 2. Class Weighting

In [10]:
class_counts = np.bincount(y_train)
weights = 1.0 / class_counts
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion_weighted = nn.CrossEntropyLoss(weight=class_weights)

model_weighted = get_model().to(device)
model_weighted = train_model(model_weighted, train_loader, criterion_weighted)

test_report_weight = evaluate(model_weighted, test_loader)

              precision    recall  f1-score   support

           0       0.97      0.87      0.91       134
           1       0.81      0.97      0.88        63
           2       0.88      0.93      0.90        40

    accuracy                           0.90       237
   macro avg       0.89      0.92      0.90       237
weighted avg       0.91      0.90      0.90       237



## 3. Oversampling

In [11]:
sample_weights = weights[y_train]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader_over = DataLoader(train_dataset, batch_size=16, sampler=sampler)

model_over = get_model().to(device)
criterion = nn.CrossEntropyLoss()

model_over = train_model(model_over, train_loader_over, criterion)

test_report_over = evaluate(model_over, test_loader)

              precision    recall  f1-score   support

           0       0.95      0.94      0.95       134
           1       0.93      0.90      0.92        63
           2       0.89      0.97      0.93        40

    accuracy                           0.94       237
   macro avg       0.93      0.94      0.93       237
weighted avg       0.94      0.94      0.94       237



## 4. Data Augmentation

In [12]:
aug_dataset = datasets.ImageFolder(DATASET_PATH, transform=aug_transform)

train_dataset_aug = Subset(aug_dataset, X_train)

train_loader_aug = DataLoader(train_dataset_aug, batch_size=16, shuffle=True)

model_aug = get_model().to(device)
criterion = nn.CrossEntropyLoss()

model_aug = train_model(model_aug, train_loader_aug, criterion)

test_report_aug = evaluate(model_aug, test_loader)

              precision    recall  f1-score   support

           0       0.94      0.97      0.96       134
           1       0.98      0.90      0.94        63
           2       0.90      0.93      0.91        40

    accuracy                           0.95       237
   macro avg       0.94      0.93      0.94       237
weighted avg       0.95      0.95      0.95       237



## 5. Focal Loss

In [13]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1-pt)**self.gamma) * ce
        
        return loss.mean()

criterion_focal = FocalLoss()

model_focal = get_model().to(device)
model_focal = train_model(model_focal, train_loader, criterion_focal)

test_report_focal = evaluate(model_focal, test_loader)

              precision    recall  f1-score   support

           0       0.95      0.89      0.92       134
           1       0.90      0.95      0.92        63
           2       0.80      0.90      0.85        40

    accuracy                           0.91       237
   macro avg       0.88      0.91      0.90       237
weighted avg       0.91      0.91      0.91       237



## Final F1 Score Comparison

In [14]:
results = pd.DataFrame({
    "Method": ["Baseline","Class Weighting","Oversampling","Augmentation","Focal Loss"],

    "Macro F1": [
        test_report_base["macro avg"]["f1-score"],
        test_report_weight["macro avg"]["f1-score"],
        test_report_over["macro avg"]["f1-score"],
        test_report_aug["macro avg"]["f1-score"],
        test_report_focal["macro avg"]["f1-score"]
    ],

    "Weighted F1": [
        test_report_base["weighted avg"]["f1-score"],
        test_report_weight["weighted avg"]["f1-score"],
        test_report_over["weighted avg"]["f1-score"],
        test_report_aug["weighted avg"]["f1-score"],
        test_report_focal["weighted avg"]["f1-score"]
    ]
})

results

,Method,Macro F1,Weighted F1
0,Baseline,0.808640,0.840319
1,Class Weighting,0.899961,0.903742
2,Oversampling,0.931765,0.936749
3,Augmentation,0.937204,0.945092
4,Focal Loss,0.896352,0.907896
